In [1]:
from ultralytics import YOLO
import os
import shutil
import torch
import config # folder names

In [2]:
# ======================================================
# 1. RESET OUTPUT FOLDERS
# ======================================================
def reset_folder(path):
    if os.path.exists(path):
        shutil.rmtree(path)
    os.makedirs(path)

reset_folder(config.PREDICTIONS)          # YOLO will write here
reset_folder(config.DATASET_UPLOAD)
reset_folder(config.DATA_READY)

# create subfolders for dataset upload
os.makedirs(f"{config.DATASET_UPLOAD}/train/images", exist_ok=True)
os.makedirs(f"{config.DATASET_UPLOAD}/train/labels", exist_ok=True)
os.makedirs(f"{config.DATASET_UPLOAD}/valid/images", exist_ok=True)   # empty
os.makedirs(f"{config.DATASET_UPLOAD}/valid/labels", exist_ok=True)   # empty
os.makedirs(f"{config.IMPORTED}", exist_ok=True)

As you can see, four folders where created: `dataset_to_upload`, `data_ready`, `predictions` and `imported_images`. 

Now you need to put your images that you want to label in the folder `imported_images`.

In [3]:
# =================================================================================
# 2. Put the images that you want to label in a folder called 'imported_images'
# =================================================================================

# Checking if this folder does not exists or if is not empty
if not os.path.exists(config.IMPORTED) or not os.listdir(config.IMPORTED):
    # Creating (again) the folder
    os.makedirs(f"{config.IMPORTED}", exist_ok=True)
    print("Please put your images on 'imported_images'")

In [4]:
# ======================================================
# 2. LOAD MODEL (always use last.pt for training)
# ======================================================
model = YOLO(f"{config.WEIGHTS}/last.pt")

In [5]:
# ======================================================
# 3. RUN INFERENCE → results go inside predictions/
# ======================================================
results = model.predict(
    source=config.IMPORTED,
    conf=0.05,
    save=True,
    save_txt=True,
    save_conf=True,
    show_conf=True,
    show_labels=True,
    project=".",
    name=config.PREDICTIONS,       # YOLO writes directly here
    exist_ok=True
)

print("🔍 Inference completed. Results saved in 'predictions/'.")


image 1/5 /Users/pc/Desktop/GreeningAI/imported_images/291.png: 384x640 51 0s, 249 1s, 164.6ms
image 2/5 /Users/pc/Desktop/GreeningAI/imported_images/292.png: 352x640 3 0s, 297 1s, 140.2ms
image 3/5 /Users/pc/Desktop/GreeningAI/imported_images/293.png: 384x640 18 0s, 282 1s, 155.2ms
image 4/5 /Users/pc/Desktop/GreeningAI/imported_images/294.png: 448x640 246 0s, 54 1s, 226.8ms
image 5/5 /Users/pc/Desktop/GreeningAI/imported_images/295.png: 480x640 281 0s, 19 1s, 676.2ms
Speed: 3.6ms preprocess, 272.6ms inference, 4.4ms postprocess per image at shape (1, 3, 480, 640)
Results saved to /Users/pc/Desktop/GreeningAI/runs/detect/predictions
5 labels saved to /Users/pc/Desktop/GreeningAI/runs/detect/predictions/labels
🔍 Inference completed. Results saved in 'predictions/'.


Now you can upload these new images to Roboflow, and **don't forget to continue labelling the missing trees!**

In [6]:
from pathlib import Path
import os
import shutil

# ======================================================
# 5. BUILD dataset_to_upload (for Roboflow)
# ======================================================

# Paths
pred_dir = Path(results[0].save_dir)
labels_dir = pred_dir / "labels"
upload_images_dir = Path(config.DATASET_UPLOAD) / "train" / "images"
upload_labels_dir = Path(config.DATASET_UPLOAD) / "train" / "labels"

# 👉 Ensure destination folders exist
upload_images_dir.mkdir(parents=True, exist_ok=True)
upload_labels_dir.mkdir(parents=True, exist_ok=True)

print("pred_dir =", pred_dir)
print("labels_dir =", labels_dir)

# ======================================================
# Copy original images
# ======================================================
copied_imgs = 0
for f in os.listdir(config.IMPORTED):
    if f.lower().endswith((".png", ".jpg", ".jpeg")):
        src = Path(config.IMPORTED) / f
        dst = upload_images_dir / f
        shutil.copy2(src, dst)  # copy2 = keeps metadata
        copied_imgs += 1

# ======================================================
# Copy YOLO labels
# ======================================================
copied_labels = 0

if labels_dir.exists():
    for f in os.listdir(labels_dir):
        if f.endswith(".txt"):
            src = labels_dir / f
            dst = upload_labels_dir / f
            shutil.copy2(src, dst)
            copied_labels += 1
else:
    print("⚠️ labels_dir does not exist:", labels_dir)

# ======================================================
print(f"Copied {copied_imgs} images.")
print(f"Copied {copied_labels} labels.")
print("dataset_to_upload/ is ready for Roboflow upload.")

pred_dir = /Users/pc/Desktop/GreeningAI/runs/detect/predictions
labels_dir = /Users/pc/Desktop/GreeningAI/runs/detect/predictions/labels
Copied 5 images.
Copied 5 labels.
dataset_to_upload/ is ready for Roboflow upload.


----
## (Optional) Improving your current model by fine-tuning it on your labelled images

After having labelled your images and uploaded them in Roboflow, you can fine-tune your current YOLO model on the whole Roboflow dataset (including the images that you just labelled) in order to make the model more performant in detecting trees, which can make the labelling process faster.

To do so, you can follow the steps explained in the notebook [fine_tuning.ipynb](fine_tuning.ipynb).

In [10]:
import cv2
from ultralytics import YOLO

# Load your fine-tuned model
model = YOLO(f"{config.WEIGHTS}/best.pt")   # change to your model path

# Input + Output videos
input_video = "input1.mp4"
output_video = "output_detected1.mp4"

# Open video
cap = cv2.VideoCapture(input_video)

# Get video metadata
fps = cap.get(cv2.CAP_PROP_FPS)
width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

# Create output writer
fourcc = cv2.VideoWriter_fourcc(*"mp4v")
out = cv2.VideoWriter(output_video, fourcc, fps, (width, height))

print("🔄 Processing video...")

while True:
    ret, frame = cap.read()
    if not ret:
        break

    # YOLO detection on frame
    results = model.predict(
        frame,
        imgsz=640,
        conf=0.25,
        save=False,           # We draw manually, no need for YOLO saving
        save_txt=False,       # Only for image-based datasets
        save_conf=True,       # Keep confidence internally if you need .txt later
        show_conf=False,      # ❌ Do not draw confidence
        show_labels=False,    # ❌ Do not draw class labels
        verbose=False
    )

    # Draw bounding boxes ONLY (YOLO handles this automatically)
    annotated_frame = results[0].plot()   # This respects show_labels=False

    # Write result frame
    out.write(annotated_frame)

# Cleanup
cap.release()
out.release()
print("✅ Done! Output saved as:", output_video)


🔄 Processing video...
✅ Done! Output saved as: output_detected1.mp4


In [7]:
import cv2
from ultralytics import YOLO
import config 
# Load your fine-tuned model
model = YOLO(f"{config.WEIGHTS}/best.pt")   # change to your model path

input_video = "input6.mp4"
output_video = "output6.mp4"

cap = cv2.VideoCapture(input_video)

fps = cap.get(cv2.CAP_PROP_FPS)
width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

fourcc = cv2.VideoWriter_fourcc(*"mp4v")
out = cv2.VideoWriter(output_video, fourcc, fps, (width, height))

while True:
    ret, frame = cap.read()
    if not ret:
        break

    results = model.predict(frame, imgsz=640, verbose=False)

    detections = results[0].boxes.xyxy.cpu().numpy()  # x1,y1,x2,y2
    # loops through boxes
    for box in detections:
        x1, y1, x2, y2 = map(int, box)
        
        # Draw bounding box only (no text)
        cv2.rectangle(frame, (x1, y1), (x2, y2), (255, 0, 0), thickness=4)

    out.write(frame)

cap.release()
out.release()
print("✅ Video saved WITHOUT confidence or labels.")


✅ Video saved WITHOUT confidence or labels.
